In [3]:
import openai
import pandas as pd
import chromadb
import os
import shutil
from openai import OpenAI
from chromadb.config import Settings
from dotenv import load_dotenv
from docx import Document
from langchain.memory import ConversationBufferMemory

In [4]:
# Load openAI api key
load_dotenv()

# Access the API key
api_key = os.getenv("API_KEY")

In [5]:
# Set OpenAI API key
client = OpenAI(api_key=api_key)

# Step 1: Extract text from a .docx file
def extract_text_from_docx(file_path):
    """Extracts and returns text from a .docx file, chunked by headings 3 and above."""
    doc = Document(file_path)
    sections = []
    current_section = []

    for paragraph in doc.paragraphs:
        # Check if the paragraph is a heading (Heading 3 or higher)
        if paragraph.style.name.startswith('Heading'):
            heading_level = int(paragraph.style.name.split(' ')[-1])
            if heading_level >= 3:  # Only chunk by Heading 3 and above
                if current_section:
                    sections.append("\n".join(current_section))
                    current_section = []  # Reset for new section
        current_section.append(paragraph.text.strip())

    # Add the last section
    if current_section:
        sections.append("\n".join(current_section))

    return sections

# Step 2: Generate embeddings using OpenAI API
def generate_embeddings(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",  # Or the appropriate embedding model
        input=text
    )
    return response.data[0].embedding

# Step 3: Initialize Persistent ChromaDB client without removing the persist directory
persist_directory = "D:/Project/Chatbot/chromadb_data2"

# Ensure the directory exists, if not, create it
if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)

# Initialize the Persistent ChromaDB client
persistentClient = chromadb.PersistentClient(path=persist_directory)

# Step 4: Create or retrieve ChromaDB collection
collection_name = "document_embeddings"
if collection_name in [col.name for col in persistentClient.list_collections()]:
    collection = persistentClient.get_collection(collection_name)
else:
    collection = persistentClient.create_collection(collection_name)

# Step 5: Add embeddings to ChromaDB
def embed_and_store(file_path):
    """Embed and store text from a .docx file in ChromaDB, chunked by headings >= Heading 3."""
    # Extract sections from .docx
    sections = extract_text_from_docx(file_path)
    print(f"Extracted text from {file_path}.")

    for idx, section in enumerate(sections):
        # Generate embedding for the section
        embedding = generate_embeddings(section.strip())
        
        # If embedding is generated successfully, store it in ChromaDB
        if embedding:
            collection.add(
                documents=[section],
                embeddings=[embedding],
                metadatas=[{"Heading": f"Section {idx + 1}"}],  # Use section index as metadata
                ids=[f"{os.path.basename(file_path)}_section_{idx + 1}"]
            )
            print(f"Processed section {idx + 1}/{len(sections)}")
    
    # Persist ChromaDB state
    print(f"Embeddings stored in {persist_directory}.")

In [ ]:
# # Embedding Company Data to store in ChromaDB
# file_path = "D:\Project\Chatbot\Company Data.docx"
# embed_and_store(file_path)

In [ ]:
# # Read the data in ChromaDB
# # Retrieve the collection
# persist_directory = "D:/Project/Chatbot/chromadb_data2"
# persistentClient = chromadb.PersistentClient(path=persist_directory)
# retrieved_collection = persistentClient.get_collection(name="document_embeddings")

# # Fetch all documents, embeddings, and metadata
# all_data = retrieved_collection.get(include=["documents", "metadatas", "embeddings"])

# # Print all retrieved information
# for i, document in enumerate(all_data["documents"]):
#     print(f"\nDocument {i + 1}:")
#     print("Document Content:", document)
#     print("Metadata:", all_data["metadatas"][i])
#     print("Embedding (first 5 values):", all_data["embeddings"][i][:5])  # Only print first 5 values for brevity

In [ ]:
# # Test query chromaDB
# query = "What is Davinci Labs"
# # Step 6: Query the ChromaDB collection
# def query_chromadb(query, n_results=5):
#     """Query ChromaDB collection for the most relevant results based on the query."""
#     # Generate embedding for the query
#     query_embedding = generate_embeddings(query)
    
#     # Query the collection for most relevant results
#     results = collection.query(
#         query_embeddings=[query_embedding],  # Query with embedding
#         n_results=n_results  # Limit to the top 'n' results
#     )
    
#     return results

# # Example: Query with the input "What is Davinci Labs?"
# query = "What is Davinci Labs"
# n_results = 3  # Limit to top 3 results
# query_results = query_chromadb(query, n_results)

# # Print results - Only the text (documents)
# print(f"Query: {query}")
# print(f"Top {n_results} results:")
# for i, result in enumerate(query_results['documents']):
#     print(f"Result {i + 1}:")
#     print(f"Text: {result}")
#     print()



Query: What is Davinci Labs
Top 3 results:
Result 1:
Text: ['DAVinCI LABS\nDAVinCI LABS automate and simplify machine learning technology and enable data analysis with just a few clicks.\nDAVinCI LABS is AI (Artificial Intelligence) based data analytics system. Until now Machine Learning technology has been occupied by only a few experts.\nIntuitive user interface that anyone can easily start analyzing by machine learning\nNo coding required, just click to adjust parameters of algorithm, you can create a variety of models.\nAutomated Machine Learning (Auto ML)\xa0is the process of automating the process of applying machine learning to real-world problems. Auto ML covers the complete pipeline from the raw dataset to the deployable machine learning model. Auto ML was proposed as an artificial intelligence-based solution to the ever-growing challenge of applying machine learning. The high degree of automation in Auto ML allows non-experts to make use of machine learning models and techniq

In [6]:
# Initialize the Persistent ChromaDB client
client_chroma = chromadb.PersistentClient()

# Retrieve the collection
collection_name = "document_embeddings"
retrieved_collection = persistentClient.get_collection(name=collection_name)

# Function to query ChromaDB for relevant documents
def query_chromadb(query, n_results=5):
    """Query ChromaDB collection for the most relevant results based on the query."""
    # Generate embedding for the query
    query_embedding = generate_embeddings(query)
    
    # Query the collection for most relevant results
    results = collection.query(
        query_embeddings=[query_embedding],  # Query with embedding
        n_results=n_results  # Limit to the top 'n' results
    )
    
    return results

# Function to interact with GPT-4 and generate a response
def generate_response(user_query, n_results=3):
    # Step 1: Query ChromaDB for relevant documents
    results = query_chromadb(user_query)
    
    # Step 2: Format the documents and metadata for GPT-4
    context = results['documents']  # Use only the document text as context
    
    # Print the context and the query before sending it to OpenAI
    print("Sending to OpenAI:")
    print("Context:\n", context)
    
    # Step 3: Generate the GPT-4 response using the retrieved context
    response = openai.chat.completions.create(  
        model="gpt-4o-mini",  # Specify the GPT-4o-mini model
        messages=[ 
            {"role": "system", "content": "You are a helpful assistant to help user answer the question about iCONEXT Company"},
            {"role": "user", "content": f"{user_query}\nContext:\n{context}\n help to answer the question by using the context and your knowledge"}
        ]
    )
    
    # Accessing the content from the response object correctly
    return response.choices[0].message.content


In [7]:
# Example 
user_query = "What is Davinci labs"
client = OpenAI(api_key=api_key)
response = generate_response(user_query)
print(response)

Sending to OpenAI:
Context:
 [['DAVinCI LABS\nDAVinCI LABS automate and simplify machine learning technology and enable data analysis with just a few clicks.\nDAVinCI LABS is AI (Artificial Intelligence) based data analytics system. Until now Machine Learning technology has been occupied by only a few experts.\nIntuitive user interface that anyone can easily start analyzing by machine learning\nNo coding required, just click to adjust parameters of algorithm, you can create a variety of models.\nAutomated Machine Learning (Auto ML)\xa0is the process of automating the process of applying machine learning to real-world problems. Auto ML covers the complete pipeline from the raw dataset to the deployable machine learning model. Auto ML was proposed as an artificial intelligence-based solution to the ever-growing challenge of applying machine learning. The high degree of automation in Auto ML allows non-experts to make use of machine learning models and techniques without requiring becomin

: 

In [26]:
# # Test adding Memory >>> Still not work
# # Initialize the Persistent ChromaDB client
# client_chroma = chromadb.PersistentClient()

# # Retrieve the collection
# collection_name = "document_embeddings"
# retrieved_collection = persistentClient.get_collection(name=collection_name)

# # Initialize memory for conversation history
# memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# # Function to query ChromaDB for relevant documents
# def query_chromadb(query, n_results=5):
#     """Query ChromaDB collection for the most relevant results based on the query."""
#     # Generate embedding for the query
#     query_embedding = generate_embeddings(query)
    
#     # Query the collection for most relevant results
#     results = collection.query(
#         query_embeddings=[query_embedding],  # Query with embedding
#         n_results=n_results  # Limit to the top 'n' results
#     )
    
#     return results

# # Function to interact with GPT-4 and generate a response
# def generate_response(user_query, n_results=3):
#     # Step 1: Query ChromaDB for relevant documents
#     results = query_chromadb(user_query)

#     # Step 2: Format the documents and metadata for GPT-4
#     context = results['documents']  # Use only the document text as context

#     # Add user query and retrieved context to memory
#     memory.chat_memory.add_user_message(user_query)
#     memory.chat_memory.add_ai_message(f"Retrieved Context: {context}")  
    
#     # Print the context and the query before sending it to OpenAI
#     print("Sending to OpenAI:")
#     print("Context:\n", context)

#     # Step 3: Prepare chat history for GPT-4
#     chat_history = memory.load_memory_variables({})["chat_history"]
    
#     # Step 4: Generate the GPT-4 response using the context and chat history
#     response = openai.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[
#             {"role": "system", "content": "You are a helpful assistant to help user answer questions about iCONEXT Company."},
#             {"role": "user", "content": f"{user_query}\n\nContext:\n{context}\n\nHelp answer the question by using the context and your knowledge."},
#             {"role": "assistant", "content": f"Conversation history:\n{chat_history}"}
#         ]
#     )
    
#     # Step 5: Add GPT-4 response to memory
#     ai_response = response.choices[0].message.content
#     memory.chat_memory.add_ai_message(ai_response)
#     return ai_response